In [ ]:
def evaluate_classifier(
    model,
    test_loader,
    device,
    class_names: list = None,
) -> dict:
    '''
    Evalúa el clasificador en test set con reportes detallados.
    Retorna diccionario con todas las métricas y visualizaciones.
    
    Args:
        model: nn.Module entrenado
        test_loader: DataLoader del test set
        device: torch.device (cuda o cpu)
        class_names: lista de nombres de clases (default CLASS_NAMES)
    
    Returns:
        dict con claves:
            - balanced_accuracy: float
            - macro_f1, weighted_f1: float
            - auc_ovr: AUC One-vs-Rest
            - precision_per_class, recall_per_class, f1_per_class: np.ndarray
            - support: cantidad de muestras por clase
            - confusion_matrix: np.ndarray NxN
            - all_preds, all_labels, all_probs: np.ndarray
            - classification_report: str
    '''
    if class_names is None:
        class_names = CLASS_NAMES
    
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for x, y in tqdm(test_loader, desc='Evaluating', leave=False):
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
    
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    
    # Métricas globales
    from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix
    from sklearn.metrics import precision_recall_fscore_support, classification_report
    
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    
    # Métricas por clase
    precision_per_class, recall_per_class, f1_per_class, support = precision_recall_fscore_support(
        all_labels, all_preds, average=None, zero_division=0
    )
    
    # Matriz de confusión
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(class_names))))
    
    # AUC One-vs-Rest
    try:
        auc_ovr = roc_auc_score(all_labels, all_probs, multi_class='ovr', zero_division=0)
    except:
        auc_ovr = np.nan
    
    # Classification report
    class_report = classification_report(
        all_labels, all_preds,
        target_names=class_names,
        zero_division=0,
    )
    
    # Visualizaciones
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Confusion Matrix Heatmap
    import seaborn as sns
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[0],
    )
    axes[0].set_title('Confusion Matrix')
    axes[0].set_ylabel('True')
    axes[0].set_xlabel('Predicted')
    
    # 2. Per-class metrics comparison
    x = np.arange(len(class_names))
    width = 0.25
    
    axes[1].bar(x - width, precision_per_class, width, label='Precision')
    axes[1].bar(x, recall_per_class, width, label='Recall')
    axes[1].bar(x + width, f1_per_class, width, label='F1')
    
    axes[1].set_ylabel('Score')
    axes[1].set_title('Per-Class Metrics')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(class_names, rotation=45, ha='right')
    axes[1].legend()
    axes[1].set_ylim([0, 1])
    
    plt.tight_layout()
    plt.show()
    
    # Reporte
    print("\n" + "="*60)
    print("EVALUATION REPORT")
    print("="*60)
    print(f"\nGlobal Metrics:")
    print(f"  Balanced Accuracy: {balanced_acc:.4f}")
    print(f"  Macro F1:          {macro_f1:.4f}")
    print(f"  Weighted F1:       {weighted_f1:.4f}")
    print(f"  AUC OvR:           {auc_ovr:.4f}")
    
    print(f"\nPer-Class Metrics:")
    for i, (name, prec, rec, f1, sup) in enumerate(
        zip(class_names, precision_per_class, recall_per_class, f1_per_class, support)
    ):
        print(f"  {name:20s}: P={prec:.3f} R={rec:.3f} F1={f1:.3f} (n={int(sup)})")
    
    print(f"\n{class_report}")
    print("="*60)
    
    return {
        'balanced_accuracy': balanced_acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'auc_ovr': auc_ovr,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'f1_per_class': f1_per_class,
        'support': support,
        'confusion_matrix': cm,
        'all_preds': all_preds,
        'all_labels': all_labels,
        'all_probs': all_probs,
        'classification_report': class_report,
    }

In [ ]:
def train_classifier(config: dict) -> Tuple[nn.Module, dict]:
    '''
    Función principal que orquesta entrenamiento completo del clasificador.
    
    1. Carga stats de U-Net (channel_means/stds/reinhard)
    2. Hace split de biopsias (igual que U-Net, seed=42)
    3. Construye manifest de instancias
    4. Crea dataloaders + modelo + optimizer + scheduler
    5. Entrena con early stopping sobre balanced_accuracy
    6. Guarda checkpoints
    7. Retorna modelo y reporte
    '''
    
    # Setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    output_dir = Path(config['output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Cargar stats de U-Net checkpoint
    print("[1/7] Loading U-Net checkpoint stats...")
    ckpt = torch.load(config['unet_checkpoint'], map_location='cpu')
    channel_means = ckpt.get('channel_means', [0.5, 0.5, 0.5])
    channel_stds = ckpt.get('channel_stds', [0.2, 0.2, 0.2])
    reinhard_stats = ckpt.get('reinhard_stats')
    print(f"  channel_means: {channel_means}, channel_stds: {channel_stds}")
    
    # 2. Split biopsias (MISMO que U-Net)
    print("[2/7] Splitting biopsias...")
    all_tiles = _collect_image_tiles(config['images_dir'])
    groups = _group_images_by_biopsy(all_tiles, config['images_dir'])
    train_biopsias, val_biopsias, test_biopsias, _ = split_biopsias(
        config['images_dir'],
        train_size=config['train_size'],
        val_size=config['val_size'],
        seed=config['seed'],
    )
    biopsias_split = {
        'train': train_biopsias,
        'val': val_biopsias,
        'test': test_biopsias,
    }
    print(f"  Train: {len(biopsias_split['train'])} biopsias")
    print(f"  Val: {len(biopsias_split['val'])} biopsias")
    print(f"  Test: {len(biopsias_split['test'])} biopsias")
    
    # 3. Build manifest
    print("[3/7] Building glomerulus manifest...")
    manifest = build_glomerulus_manifest(
        images_dir=config['images_dir'],
        masks_dir=config['masks_dir'],
        split_biopsias_result=biopsias_split,
        min_area_px=config['min_area_px'],
        min_distance=config['min_distance'],
        output_csv=str(output_dir / 'manifest.csv'),
    )
    print(f"  Total instances: {len(manifest)}")
    print(f"  Class distribution:\n{manifest['class_id'].value_counts().sort_index()}")
    
    # 4. Reinhard normalization
    print("[4/7] Setting up preprocessing...")
    reinhard_norm = ReinhardNormalize(reinhard_stats) if reinhard_stats else None
    
    # 5. DataLoaders
    print("[5/7] Creating dataloaders...")
    train_loader, val_loader, test_loader = create_classification_dataloaders(
        manifest,
        config,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
    )
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches: {len(val_loader)}")
    print(f"  Test batches: {len(test_loader)}")
    
    # 6. Model + Optimizer + Scheduler
    print("[6/7] Setting up model & optimizer...")
    model = build_classifier(
        backbone=config['backbone'],
        num_classes=config['num_classes'],
        in_chans=config['in_chans'],
        pretrained=config['pretrained'],
    ).to(device)
    
    criterion = build_criterion(manifest, device)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay'],
    )
    
    # Scheduler: warmup lineal + cosine annealing
    warmup = LinearLR(
        optimizer,
        start_factor=0.1,
        end_factor=1.0,
        total_iters=config['warmup_epochs'],
    )
    cosine = CosineAnnealingLR(
        optimizer,
        T_max=config['epochs'] - config['warmup_epochs'],
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup, cosine],
        milestones=[config['warmup_epochs']],
    )
    
    scaler = GradScaler() if config['use_amp'] else None
    
    # 7. Training loop
    print("[7/7] Training...")
    best_val_balanced_acc = -1
    patience_counter = 0
    history = {
        'train_loss': [], 'val_loss': [],
        'val_balanced_acc': [], 'val_macro_f1': [],
    }
    
    for epoch in range(config['epochs']):
        # Train
        train_loss = train_epoch_clf(
            model, train_loader, criterion, optimizer, device,
            scaler=scaler, epoch=epoch,
        )
        
        # Validate
        val_loss, val_metrics = eval_epoch_clf(
            model, val_loader, criterion, device, split_name='val'
        )
        
        val_balanced_acc = val_metrics['balanced_accuracy']
        val_macro_f1 = val_metrics['macro_f1']
        
        # Logging
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_balanced_acc'].append(val_balanced_acc)
        history['val_macro_f1'].append(val_macro_f1)
        
        print(f"Epoch {epoch+1:3d}/{config['epochs']} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Bal.Acc: {val_balanced_acc:.4f} | "
              f"Val Macro F1: {val_macro_f1:.4f}")
        
        # Early stopping + checkpoint
        if val_balanced_acc > best_val_balanced_acc + config['early_stopping_min_delta']:
            best_val_balanced_acc = val_balanced_acc
            patience_counter = 0
            
            # Guardar best checkpoint
            ckpt_path = output_dir / 'best_classifier.pth'
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict': scaler.state_dict() if scaler else None,
                'best_val_balanced_acc': best_val_balanced_acc,
                'class_names': CLASS_NAMES,
                'channel_means': channel_means,
                'channel_stds': channel_stds,
                'reinhard_stats': reinhard_stats,
                'config': config,
            }, ckpt_path)
            print(f"  → Checkpoint saved to {ckpt_path}")
        else:
            patience_counter += 1
            if patience_counter >= config['early_stopping_patience']:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break
        
        scheduler.step()
    
    # Evaluate on test set
    print("\nEvaluating on test set...")
    test_loss, test_metrics = eval_epoch_clf(
        model, test_loader, criterion, device, split_name='test'
    )
    
    # Reporte final
    report = {
        'history': history,
        'test_metrics': test_metrics,
        'test_loss': test_loss,
        'best_val_balanced_acc': best_val_balanced_acc,
    }
    
    return model, report

In [ ]:
def build_glomerulus_manifest(
    images_dir: str,
    masks_dir: str,
    split_biopsias_result: dict,  # {'train': [...], 'val': [...], 'test': [...]}
    min_area_px: int = 1500,
    min_distance: int = 15,
    output_csv: str = None,
) -> pd.DataFrame:
    """
    Enumerate ALL glomerulus instances in the dataset (train+val+test).
    Each row is one glomerulus instance with its metadata, bbox, and class.
    
    Algorithm:
    1. For each split ('train', 'val', 'test'):
       - For each biopsia in split:
         - Enumerate tiles: {images_dir}/{biopsia}/images/*.png
         - For EACH tile:
           a. Load mask_gray = load_binary_mask(mask_path, binary=False)
              (PNG uint8 with values 0/64/128/192/255)
           b. Binarize: binary_mask = (mask_gray > 0).astype(float)
           c. Run instance detection: instances = postprocess_prob_to_instances(...)
           d. For EACH instance:
              - Extract bbox: instance.bbox → (r1, c1, r2, c2)
              - Extract region from mask_gray: mask_crop = mask_gray[r1:r2, c1:c2]
              - Find dominant (most frequent) value: dominant = bincount(mask_crop.flatten()).argmax()
              - Map to class: class_id = GRAY_TO_CLASS[dominant] (skip if -1/background)
              - Add row with: slide_id, tile_path, mask_path, instance_id, class_name, 
                             class_id, gray_value, bbox coords, split
    
    2. Return DataFrame with all rows
    3. If output_csv not None, save to CSV
    
    Args:
        images_dir: Root directory with {biopsia}/images/*.png structure
        masks_dir: Root directory with {biopsia}/masks/*_mask.png structure
        split_biopsias_result: dict with keys 'train', 'val', 'test' → list of biopsia names
        min_area_px: Minimum instance area in pixels (default 1500)
        min_distance: Minimum distance between watershed peaks (default 15)
        output_csv: If not None, save manifest to this CSV path
    
    Returns:
        pd.DataFrame with columns:
        - slide_id: biopsia name
        - tile_path: absolute path to tile PNG
        - mask_path: absolute path to mask PNG
        - instance_id: index within tile (0, 1, 2, ...)
        - class_name: CLASS_NAMES[class_id]
        - class_id: int 0-3
        - gray_value: dominant gray value (64, 128, 192, 255)
        - bbox_r1, bbox_c1, bbox_r2, bbox_c2: bounding box coordinates
        - split: 'train'/'val'/'test'
    
    Validation:
        - DataFrame has ~100s-1000s rows (1 per glomerulus, not per tile)
        - All columns present
        - No NaN in critical columns
        - Split distribution: ~70% train, ~15% val, ~15% test (approximate)
    """
    images_dir = Path(images_dir)
    masks_dir = Path(masks_dir)
    
    manifest_rows = []
    
    # Iterate through all splits
    for split_name in ['train', 'val', 'test']:
        biopsias = split_biopsias_result.get(split_name, [])
        
        for biopsia in biopsias:
            # Find all tiles in this biopsia: {images_dir}/{biopsia}/images/*.png
            tiles_glob = sorted((images_dir / biopsia / 'images').glob('*.png'))
            
            for tile_path in tiles_glob:
                # Construct corresponding mask path
                tile_stem = tile_path.stem
                mask_path = masks_dir / biopsia / 'masks' / f'{tile_stem}_mask.png'
                
                # Check if mask exists
                if not mask_path.exists():
                    warnings.warn(f"Mask not found for tile {tile_path}, skipping.")
                    continue
                
                # Load mask as grayscale (uint8, not binarized)
                try:
                    mask_gray = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                    if mask_gray is None:
                        warnings.warn(f"Failed to load mask: {mask_path}")
                        continue
                except Exception as e:
                    warnings.warn(f"Error loading mask {mask_path}: {e}")
                    continue
                
                # Binarize: any non-zero value is glomerulus
                binary_mask = (mask_gray > 0).astype(float)
                
                # Run instance detection
                instances = postprocess_prob_to_instances(
                    binary_mask,
                    threshold=0.5,
                    min_area_px=min_area_px,
                    min_distance=min_distance,
                )
                
                # For each detected instance, extract metadata
                for instance_id, instance in enumerate(instances):
                    # Extract bounding box
                    r1, c1, r2, c2 = instance.bbox
                    
                    # Extract the region from mask_gray
                    mask_crop = mask_gray[r1:r2, c1:c2]
                    
                    # Find dominant (most frequent) gray value
                    flat = mask_crop.flatten()
                    bincount = np.bincount(flat)
                    dominant = np.argmax(bincount)
                    
                    # Map to class
                    class_id = GRAY_TO_CLASS.get(int(dominant), -1)
                    
                    # Skip background/unknown classes
                    if class_id == -1:
                        continue
                    
                    class_name = CLASS_NAMES[class_id]
                    
                    # Add row to manifest
                    manifest_rows.append({
                        'slide_id': biopsia,
                        'tile_path': str(tile_path.absolute()),
                        'mask_path': str(mask_path.absolute()),
                        'instance_id': instance_id,
                        'class_name': class_name,
                        'class_id': class_id,
                        'gray_value': int(dominant),
                        'bbox_r1': int(r1),
                        'bbox_c1': int(c1),
                        'bbox_r2': int(r2),
                        'bbox_c2': int(c2),
                        'split': split_name,
                    })
    
    # Build DataFrame
    df = pd.DataFrame(manifest_rows)
    
    # Validation
    if df.empty:
        warnings.warn("No glomerulus instances found. Manifest is empty.")
        return df
    
    # Check for NaN in critical columns
    critical_cols = ['slide_id', 'tile_path', 'mask_path', 'instance_id', 
                     'class_id', 'gray_value', 'split']
    for col in critical_cols:
        if col not in df.columns:
            raise ValueError(f"Missing critical column: {col}")
        if df[col].isna().any():
            raise ValueError(f"Found NaN values in critical column: {col}")
    
    # Log statistics
    split_counts = df['split'].value_counts()
    print(f"\nGlomerulus Manifest Summary:")
    print(f"  Total instances: {len(df)}")
    print(f"  Split distribution:")
    for split in ['train', 'val', 'test']:
        count = split_counts.get(split, 0)
        pct = 100.0 * count / len(df) if len(df) > 0 else 0.0
        print(f"    {split}: {count} ({pct:.1f}%)")
    
    class_counts = df['class_id'].value_counts().sort_index()
    print(f"  Class distribution:")
    for class_id in sorted(class_counts.index):
        count = class_counts[class_id]
        pct = 100.0 * count / len(df)
        class_name = CLASS_NAMES[class_id]
        print(f"    {class_id} ({class_name}): {count} ({pct:.1f}%)")
    
    # Save to CSV if requested
    if output_csv is not None:
        output_path = Path(output_csv)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_path, index=False)
        print(f"\n  ✓ Manifest saved to {output_path}")
    
    return df


In [ ]:
# Standard library
import os
import json
import time
from pathlib import Path
from datetime import datetime
from typing import Tuple
import random
import re

# Scientific computing
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
import albumentations as A

# Data loading
import cv2
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, WeightedRandomSampler
import warnings

# Progress/model library imports
from tqdm.auto import tqdm
import segmentation_models_pytorch as smp
from PIL import Image, ImageFile
from skimage.measure import label, regionprops
from scipy import ndimage as _ndi
from skimage.segmentation import watershed as _watershed
from skimage.feature import peak_local_max as _plm
import matplotlib.pyplot as plt
import glob
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from scipy.ndimage import label as scipy_label, distance_transform_edt
from matplotlib import patches
from skimage.measure import find_contours
import pandas as pd
import timm

# Allow PIL to load truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# --- Default threshold values for postprocessing ---
DEFAULT_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]
# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# --- CUDA performance tuning (T4 Tensor Cores) ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True          # auto-tune conv algorithms
    torch.backends.cuda.matmul.allow_tf32 = True   # TF32 for matmul (T4 supports it)
    torch.backends.cudnn.allow_tf32 = True         # TF32 for cudnn convolutions
    print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}")
    print(f"CUDA matmul TF32: {torch.backends.cuda.matmul.allow_tf32}")
    print(f"cudnn TF32: {torch.backends.cudnn.allow_tf32}")
    print(f"PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True")

In [ ]:
GRAY_TO_CLASS = {0: -1, 64: 0, 128: 1, 192: 2, 255: 3}  # -1 = background, ignoradoCLASS_TO_GRAY = {0: 64, 1: 128, 2: 192, 3: 255}CLASS_NAMES = ["No_Proliferativo", "Proliferativo", "Esclerosado", "Excluido"]NUM_CLASSES = 4CONFIG = {    # Paths — leer desde Entradas/ (tiles crudos + máscaras manuales)    'images_dir': 'Entradas',              # <slide>/images/*.png — igual que U-Net    'masks_dir':  'Entradas',              # <slide>/masks/*_mask.png — máscaras multiclase    'output_dir': 'Salidas/Clasificador',    'unet_checkpoint': 'Salidas/best_model.pth',  # para cargar channel_means/stds/reinhard    # Split — MISMO que U-Net    'train_size': 0.70,    'val_size':   0.15,    'seed':       42,    # Crop/reconstruction    'input_size':   224,    'mask_size':    224,    'margin_ratio': 0.25,    'min_area_px':  1500,    'min_distance': 15,    # Model    'backbone':    'efficientnet_b0',    'in_chans':    4,           # RGB (3) + máscara binaria (1)    'num_classes': 4,    'pretrained':  True,    # Training    'epochs':      60,    'batch_size':  32,    'lr':          1e-3,    'weight_decay': 1e-4,    'warmup_epochs': 5,    'use_amp':     True,    'early_stopping_patience': 10,    'early_stopping_min_delta': 1e-4,    # Augmentation adjustments for classification    'elastic_p':     0.10,   # reducido vs. segmentación (0.25)    'grid_dist_p':   0.10,   # reducido    'mask_perturb_prob': 0.50,  # simula ruido de U-Net}

In [ ]:
# Standard library
import os
import json
import time
from pathlib import Path
from datetime import datetime
from typing import Tuple
import random
import re

# Scientific computing
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
import albumentations as A

# Data loading
import cv2
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, WeightedRandomSampler
import warnings

# Progress/model library imports
from tqdm.auto import tqdm
import segmentation_models_pytorch as smp
from PIL import Image, ImageFile
from skimage.measure import label, regionprops
from scipy import ndimage as _ndi
from skimage.segmentation import watershed as _watershed
from skimage.feature import peak_local_max as _plm
import matplotlib.pyplot as plt
import glob
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from scipy.ndimage import label as scipy_label, distance_transform_edt
from matplotlib import patches
from skimage.measure import find_contours
import pandas as pd

# Allow PIL to load truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# --- Default threshold values for postprocessing ---
DEFAULT_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]
# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# --- CUDA performance tuning (T4 Tensor Cores) ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True          # auto-tune conv algorithms
    torch.backends.cuda.matmul.allow_tf32 = True   # TF32 for matmul (T4 supports it)
    torch.backends.cudnn.allow_tf32 = True         # TF32 for cudnn convolutions
    print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}")
    print(f"CUDA matmul TF32: {torch.backends.cuda.matmul.allow_tf32}")
    print(f"cudnn TF32: {torch.backends.cudnn.allow_tf32}")
    print(f"PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True")


# --- Preprocessing Transforms (Reinhard Normalization + Z-score) ---

class ReinhardNormalize:
    """Reinhard stain normalization in LAB color space for consistency across slides."""

    def __init__(self, target_stats: dict):
        self.target = target_stats

    @staticmethod
    def _get_tissue_mask(img_bgr: np.ndarray) -> np.ndarray:
        """Isolate tissue pixels from background and artifacts via luminance and saturation thresholds."""
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        mask = (lab[:, :, 0] < 230) & (hsv[:, :, 1] > 10)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
        mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        return mask.astype(bool)

    @staticmethod
    def compute_template_stats(image_paths: list, n_samples: int = 200) -> dict:
        """Compute median LAB statistics from tissue pixels to define normalization target."""
        paths_list = list(image_paths)[:n_samples]
        sample = random.sample(paths_list, min(n_samples, len(paths_list)))
        all_stats = []
        
        for p in sample:
            img = cv2.imread(str(p))
            if img is None:
                continue
            tissue = ReinhardNormalize._get_tissue_mask(img)
            if tissue.sum() < 100:
                continue
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
            stats = ([lab[..., c][tissue].mean() for c in range(3)] +
                     [lab[..., c][tissue].std()  for c in range(3)])
            all_stats.append(stats)
        
        if not all_stats:
            return {'mean_L': 50, 'mean_a': 128, 'mean_b': 128,
                    'std_L': 10, 'std_a': 10, 'std_b': 10}
        
        arr = np.array(all_stats)
        keys = ['mean_L', 'mean_a', 'mean_b', 'std_L', 'std_a', 'std_b']
        return {k: float(np.median(arr[:, i])) for i, k in enumerate(keys)}

    def __call__(self, img_bgr: np.ndarray) -> np.ndarray:
        """Normalize image to match template statistics, preserving background pixels."""
        tissue = self._get_tissue_mask(img_bgr)
        if tissue.sum() < 100:
            return img_bgr
        
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        
        src = {c: (lab[..., i][tissue].mean(), lab[..., i][tissue].std())
               for i, c in enumerate(['L', 'a', 'b'])}
        
        result = lab.copy()
        for i, c in enumerate(['L', 'a', 'b']):
            m, s = src[c]
            result[..., i] = ((lab[..., i] - m) *
                              (self.target[f'std_{c}'] / (s + 1e-5)) +
                              self.target[f'mean_{c}'])
        
        result[~tissue] = lab[~tissue]
        result = np.clip(result, 0, 255).astype(np.uint8)
        return cv2.cvtColor(result, cv2.COLOR_LAB2BGR)


def compute_channel_stats(
    image_paths: list,
    n_samples: int = 200,
    reinhard_norm=None,
    std_floor: float = 0.03,
) -> Tuple[list, list]:
    """Compute weighted per-channel RGB stats from tissue pixels for Z-score normalization."""
    paths_list = list(image_paths)
    sample = random.sample(paths_list, min(n_samples, len(paths_list)))
    
    tile_means, tile_vars, tile_counts = [], [], []
    skipped_count = 0
    for p in sample:
        img_bgr = cv2.imread(str(p))
        if img_bgr is None:
            skipped_count += 1
            continue
        tissue = ReinhardNormalize._get_tissue_mask(img_bgr)
        if tissue.sum() < 100:
            skipped_count += 1
            continue

        if reinhard_norm is not None:
            img_bgr = reinhard_norm(img_bgr)

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        pixels = img_rgb[tissue]
        tile_means.append(pixels.mean(0))
        tile_vars.append(pixels.var(0))
        tile_counts.append(tissue.sum())
    
    if not tile_means:
        print(f"  ⚠️  WARNING: No valid tissue pixels found in {len(sample)} samples!")
        print(f"      {skipped_count} samples were skipped (tissue.sum() < 100 or read failed)")
        print(f"      Falling back to default values [0.5, 0.5, 0.5] and [0.2, 0.2, 0.2]")
        print(f"      This will make Z-score normalization INEFFECTIVE.")
        return [0.5, 0.5, 0.5], [0.2, 0.2, 0.2]
    
    print(f"  ✓ Computed stats from {len(tile_means)} valid samples (skipped {skipped_count})")
    
    means_arr = np.array(tile_means)
    vars_arr = np.array(tile_vars)
    counts_arr = np.array(tile_counts, dtype=np.float64)
    w = counts_arr / counts_arr.sum()
    
    mean = (means_arr * w[:, None]).sum(0)
    var = ((vars_arr + (means_arr - mean) ** 2) * w[:, None]).sum(0)
    std = np.sqrt(var)
    std = np.maximum(std, std_floor)
    return mean.tolist(), std.tolist()
def _collect_image_tiles(images_dir: str) -> list:
    """Collect input PNG tiles while excluding masks and generated mask files."""
    images_dir = Path(images_dir)
    image_paths = sorted(images_dir.glob('*/images/*.png'))

    # Fallback for flat/custom datasets: include PNGs except anything inside a masks folder
    # or files already named *_mask.png.
    if not image_paths:
        image_paths = sorted(
            p for p in images_dir.rglob('*.png')
            if 'masks' not in p.relative_to(images_dir).parts
            and not p.stem.endswith('_mask')
        )

    return image_paths


def _slide_name_from_image_path(image_path, images_dir) -> str:
    """Infer the biopsy/slide name from a tile path under <root>/<slide>/images/*.png."""
    image_path = Path(image_path)
    images_dir = Path(images_dir)
    try:
        rel = image_path.relative_to(images_dir)
        if len(rel.parts) >= 3 and rel.parts[1] == 'images':
            return rel.parts[0]
    except ValueError as e:
        warnings.warn(f'Path format issue for {image_path}: {e}')
        return None

    if image_path.parent.name == 'images' and image_path.parent.parent.name:
        return image_path.parent.parent.name
    return image_path.parent.name or image_path.stem


def _group_images_by_biopsy(image_paths: list, images_dir: str) -> dict:
    """Group image paths by biopsy/slide folder, supporting canonical and flat layouts."""
    images_dir = Path(images_dir)
    biopsias_dict = {}
    for img_path in image_paths:
        biopsia = _slide_name_from_image_path(img_path, images_dir)
        biopsias_dict.setdefault(biopsia, []).append(img_path)
    return biopsias_dict


def _safe_train_val_test_split(
    biopsias_list: list,
    train_size: float = 0.70,
    val_size: float = 0.15,
    seed: int = 42,
) -> Tuple[list, list, list]:
    """Split biopsy IDs without crashing on tiny datasets."""
    test_size = 1.0 - train_size - val_size
    assert test_size >= 0, "train_size + val_size must be <= 1.0"

    biopsias_list = list(biopsias_list)
    if not biopsias_list:
        return [], [], []

    # sklearn.train_test_split raises on very small lists. Keep deterministic, leak-free
    # biopsy-level splits and prefer having train data in quick/smoke-test datasets.
    if len(biopsias_list) == 1:
        return biopsias_list, [], []
    if len(biopsias_list) == 2:
        rng = random.Random(seed)
        shuffled = biopsias_list[:]
        rng.shuffle(shuffled)
        return [shuffled[0]], [shuffled[1]], []

    if test_size > 0:
        train_val_biopsias, test_biopsias = train_test_split(
            biopsias_list,
            test_size=test_size,
            random_state=seed,
        )
    else:
        train_val_biopsias = biopsias_list
        test_biopsias = []

    if val_size > 0 and len(train_val_biopsias) > 1:
        val_fraction = val_size / (train_size + val_size)
        train_biopsias, val_biopsias = train_test_split(
            train_val_biopsias,
            test_size=val_fraction,
            random_state=seed + 1,
        )
    else:
        train_biopsias = train_val_biopsias
        val_biopsias = []

    return train_biopsias, val_biopsias, test_biopsias


def split_biopsias(
    images_dir: str,
    train_size: float = 0.70,
    val_size: float = 0.15,
    seed: int = 42,
) -> Tuple[list, list, list, dict]:
    """Groups biopsias into train/val/test to prevent data leakage at slide level."""
    images_dir = Path(images_dir)
    all_images = _collect_image_tiles(images_dir)

    if not all_images:
        raise ValueError(f"No PNG image tiles found in {images_dir}. Expected files under */images/*.png")

    biopsias_dict = _group_images_by_biopsy(all_images, images_dir)
    train_biopsias, val_biopsias, test_biopsias = _safe_train_val_test_split(
        list(biopsias_dict.keys()),
        train_size=train_size,
        val_size=val_size,
        seed=seed,
    )

    return train_biopsias, val_biopsias, test_biopsias, biopsias_dict


def load_rgb_image(image_path: str) -> np.ndarray:
    """Load an RGB uint8 image with PIL tolerance for truncated files."""
    try:
        return np.array(Image.open(str(image_path)).convert('RGB'))
    except Exception as exc:
        raise RuntimeError(f"Failed to load image {image_path}: {exc}") from exc


def load_binary_mask(mask_path: str) -> np.ndarray:
    """Load a grayscale mask and binarize all non-zero classes as glomerulus."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise RuntimeError(f"Failed to load mask: {mask_path}")
    return (mask > 0).astype(np.uint8)


def preprocess_rgb_image(
    img_rgb: np.ndarray,
    reinhard_norm=None,
    channel_means: list = None,
    channel_stds: list = None,
) -> np.ndarray:
    """Apply the same RGB -> Reinhard -> RGB/255 -> Z-score preprocessing used by the dataset."""
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    if reinhard_norm is not None:
        img_bgr = reinhard_norm(img_bgr)

    img_float = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    if channel_means is not None and channel_stds is not None:
        means = np.asarray(channel_means, dtype=np.float32)
        stds = np.asarray(channel_stds, dtype=np.float32)
        img_float = (img_float - means) / (stds + 1e-6)
    return img_float


def tensor_from_preprocessed_rgb(img_float: np.ndarray, device: torch.device = None) -> torch.Tensor:
    """Convert preprocessed HWC RGB float image to a BCHW float tensor."""
    tensor = torch.from_numpy(np.transpose(img_float, (2, 0, 1))).float().unsqueeze(0)
    return tensor.to(device) if device is not None else tensor


def make_mask_overlay(img_rgb: np.ndarray, mask_binary: np.ndarray, color=(255, 0, 0), alpha: float = 0.4) -> np.ndarray:
    """Blend a binary mask over an RGB image."""
    overlay = img_rgb.copy().astype(np.float32)
    overlay[mask_binary.astype(bool)] = color
    return cv2.addWeighted(img_rgb, 1 - alpha, overlay.astype(np.uint8), alpha, 0)


# ============================================================================
# Grayscale pixel values that correspond to glomerulus classes (any > 0 in practice)
# 64=No_Proliferativo, 128=Proliferativo, 192=Esclerosado, 255=Excluido/Excluyente
# Excluido (255) is INTENTIONALLY mapped to Glomerulus class 1 for binary segmentation
# ============================================================================
class GlomeruliDataset(Dataset):
    """Loads paired image-mask glomeruli tiles with online preprocessing and augmentation."""

    def __init__(
        self,
        images_dir: str,
        masks_dir: str = None,
        split: str = 'train',
        biopsias: list = None,
        biopsias_dict: dict = None,
        reinhard_norm=None,
        channel_means: list = None,
        channel_stds: list = None,
        train_size: float = 0.70,
        val_size: float = 0.15,
        seed: int = 42,
        transforms=None,
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir) if masks_dir is not None else Path(images_dir)
        self.split = split
        self.transforms = transforms
        self.reinhard_norm = reinhard_norm
        self.channel_means = channel_means if channel_means is not None else [0.5, 0.5, 0.5]
        self.channel_stds = channel_stds if channel_stds is not None else [0.2, 0.2, 0.2]

        assert split in {'train', 'val', 'test'}, f"Invalid split: {split}"
        assert self.images_dir.exists(), f"Images dir not found: {self.images_dir}"
        assert self.masks_dir.exists(), f"Masks dir not found: {self.masks_dir}"

        if biopsias is not None and biopsias_dict is not None:
            selected_biopsias = biopsias
            self.biopsias_dict = biopsias_dict
        else:
            train_biopsias, val_biopsias, test_biopsias, self.biopsias_dict = split_biopsias(
                self.images_dir,
                train_size=train_size,
                val_size=val_size,
                seed=seed,
            )
            selected_biopsias = {
                'train': train_biopsias,
                'val': val_biopsias,
                'test': test_biopsias,
            }[split]

        self.image_paths = []
        for biopsia in selected_biopsias:
            self.image_paths.extend(self.biopsias_dict.get(biopsia, []))

        self.image_paths = sorted(self.image_paths)

        paired = []
        missing = []
        for img_path in self.image_paths:
            mask_path = self._get_mask_path(img_path)
            if mask_path.exists():
                paired.append((img_path, mask_path))
            else:
                missing.append((img_path, mask_path))

        if missing:
            warnings.warn(
                f"Found {len(missing)} images without corresponding masks. "
                f"These will be skipped. First few: {missing[:3]}"
            )

        if not paired:
            raise ValueError("No valid image-mask pairs found after checking.")

        self.image_paths, self.mask_paths = zip(*paired)
        self.image_paths = list(self.image_paths)
        self.mask_paths = list(self.mask_paths)
        
        # Caches used by the train sampler/audits
        self._positive_flags = None
        self._tile_metadata = None
        self._annotation_tiles_by_key = None

    def _get_mask_path(self, image_path: Path) -> Path:
        """Convert image tile path to its mask path, supporting canonical and flat layouts."""
        rel = image_path.relative_to(self.images_dir)
        parts = list(rel.parts)
        stem = Path(parts[-1]).stem

        if len(parts) >= 3 and parts[1] == 'images':
            parts[1] = 'masks'
            parts[-1] = f"{stem}_mask.png"
            return self.masks_dir / Path(*parts)

        # Flat/custom fallback: first try <masks_dir>/<stem>_mask.png, then masks/<stem>_mask.png.
        flat_mask = self.masks_dir / f"{stem}_mask.png"
        if flat_mask.exists():
            return flat_mask
        return self.masks_dir / 'masks' / f"{stem}_mask.png"

    def get_positive_flags(self) -> list:
        """Return list of bools: True if tile mask contains at least one glomerulus pixel.
        
        Used by WeightedRandomSampler. Reads mask files once at dataset init time.
        Masks are small enough (1024x1024 uint8 = 1MB) that this is feasible.
        Caches result to avoid double scan.
        """
        if self._positive_flags is not None:
            return self._positive_flags
        
        flags = []
        for mask_path in self.mask_paths:
            try:
                flags.append(bool(np.any(load_binary_mask(mask_path))))
            except RuntimeError:
                flags.append(False)
        self._positive_flags = flags
        return flags

    def _load_annotation_tiles_by_key(self) -> dict:
        """Index per-slide annotations.json entries by (slide_folder, tile image path)."""
        if self._annotation_tiles_by_key is not None:
            return self._annotation_tiles_by_key

        tiles_by_key = {}
        for ann_path in sorted(self.images_dir.glob('*/annotations.json')):
            slide_folder = ann_path.parent.name
            try:
                with ann_path.open('r', encoding='utf-8') as f:
                    annotations = json.load(f)
            except Exception as exc:
                warnings.warn(f"Could not read annotations metadata from {ann_path}: {exc}")
                continue

            slide_name = annotations.get('slide') or slide_folder
            for tile in annotations.get('tiles', []):
                image_rel = tile.get('image')
                if not image_rel:
                    continue

                image_rel = str(Path(image_rel).as_posix())
                meta = dict(tile)
                meta['slide'] = slide_name
                meta['slide_folder'] = slide_folder
                meta['annotations_path'] = str(ann_path)

                # Support both the folder name and the slide name in case they differ.
                tiles_by_key[(slide_folder, image_rel)] = meta
                tiles_by_key[(slide_name, image_rel)] = meta

        self._annotation_tiles_by_key = tiles_by_key
        return tiles_by_key

    def get_tile_metadata(self, idx: int) -> dict:
        """Return annotations.json metadata for a dataset tile, or {} if unavailable."""
        if self._tile_metadata is None:
            tiles_by_key = self._load_annotation_tiles_by_key()
            metadata = []
            for image_path in self.image_paths:
                try:
                    rel = image_path.relative_to(self.images_dir)
                    slide_folder = rel.parts[0]
                    image_rel = Path(*rel.parts[1:]).as_posix()
                except Exception:
                    metadata.append({})
                    continue

                metadata.append(tiles_by_key.get((slide_folder, image_rel), {}))

            self._tile_metadata = metadata

        return self._tile_metadata[idx] if 0 <= idx < len(self._tile_metadata) else {}

    def get_sampling_weights(
        self,
        secondary_factor: float = 0.35,
        duplicate_aware: bool = True,
    ) -> Tuple[torch.Tensor, dict]:
        """Compute train-sampling weights with optional duplicate-aware glomerulus balancing."""
        positive_flags = self.get_positive_flags()
        n_positive = int(sum(positive_flags))
        n_negative = int(len(positive_flags) - n_positive)

        report = {
            'mode': 'simple',
            'n_positive': n_positive,
            'n_negative': n_negative,
            'num_unique_glomeruli': 0,
            'primary_links': 0,
            'secondary_links': 0,
            'unmatched_positive_tiles': 0,
        }

        if n_positive == 0 or n_negative == 0:
            return None, report

        def simple_weights(mode: str = 'simple'):
            report['mode'] = mode
            weight_pos = 1.0 / n_positive
            weight_neg = 1.0 / n_negative
            return torch.tensor(
                [weight_pos if flag else weight_neg for flag in positive_flags],
                dtype=torch.float32,
            ), report

        if not duplicate_aware:
            return simple_weights('simple')

        glomerulus_entries = {}
        for idx, is_positive in enumerate(positive_flags):
            if not is_positive:
                continue

            meta = self.get_tile_metadata(idx)
            glomeruli = meta.get('glomeruli') if meta else None
            if not glomeruli:
                continue

            slide = meta.get('slide') or meta.get('slide_folder') or _slide_name_from_image_path(
                self.image_paths[idx], self.images_dir
            )
            for glom in glomeruli:
                glom_id = glom.get('id')
                if glom_id is None:
                    continue

                role = str(glom.get('role', 'primary')).lower()
                try:
                    coverage = max(float(glom.get('coverage_pct', 0.0)), 0.0) / 100.0
                except (TypeError, ValueError):
                    coverage = 0.0

                role_factor = secondary_factor if role == 'secondary' else 1.0
                raw_weight = max(coverage, 1e-6) * role_factor
                key = (str(slide), str(glom_id))
                glomerulus_entries.setdefault(key, []).append((idx, raw_weight, role))

                if role == 'secondary':
                    report['secondary_links'] += 1
                else:
                    report['primary_links'] += 1

        if not glomerulus_entries:
            warnings.warn(
                "Duplicate-aware sampler requested, but no usable glomerulus metadata was found. "
                "Falling back to simple positive/negative balancing."
            )
            return simple_weights('fallback-simple-no-annotations')

        positive_mass = np.zeros(len(positive_flags), dtype=np.float64)
        for entries in glomerulus_entries.values():
            total = sum(raw for _, raw, _ in entries)
            if total <= 0:
                continue
            for idx, raw, _ in entries:
                positive_mass[idx] += raw / total

        unmatched_positive_tiles = 0
        for idx, is_positive in enumerate(positive_flags):
            if is_positive and positive_mass[idx] <= 0:
                # Keep mask-positive tiles with missing/partial metadata trainable.
                positive_mass[idx] = 1.0
                unmatched_positive_tiles += 1

        total_positive_mass = float(positive_mass.sum())
        if total_positive_mass <= 0:
            warnings.warn(
                "Duplicate-aware sampler produced zero positive mass. "
                "Falling back to simple positive/negative balancing."
            )
            return simple_weights('fallback-simple-zero-positive-mass')

        weights = np.zeros(len(positive_flags), dtype=np.float64)
        for idx, is_positive in enumerate(positive_flags):
            if is_positive:
                weights[idx] = positive_mass[idx] / total_positive_mass
            else:
                weights[idx] = 1.0 / n_negative

        report.update({
            'mode': 'duplicate-aware',
            'num_unique_glomeruli': len(glomerulus_entries),
            'unmatched_positive_tiles': unmatched_positive_tiles,
            'positive_weight_sum': float(weights[np.array(positive_flags, dtype=bool)].sum()),
            'negative_weight_sum': float(weights[~np.array(positive_flags, dtype=bool)].sum()),
            'secondary_factor': secondary_factor,
        })
        return torch.tensor(weights, dtype=torch.float32), report

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Load RGB tile -> Reinhard normalization -> augment -> Z-score -> tensors."""
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        img_rgb = load_rgb_image(img_path)
        mask_binary = load_binary_mask(mask_path)

        if self.reinhard_norm is not None:
            img_bgr = self.reinhard_norm(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        if self.transforms is not None:
            augmented = self.transforms(image=img_rgb, mask=mask_binary)
            img_rgb = augmented['image']
            mask_binary = augmented['mask']

        img_float = preprocess_rgb_image(
            img_rgb,
            reinhard_norm=None,  # already applied before augmentation when configured
            channel_means=self.channel_means,
            channel_stds=self.channel_stds,
        )
        img_tensor = torch.from_numpy(np.transpose(img_float, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(mask_binary.astype(np.int64)).long()

        return img_tensor, mask_tensor
# Augmentation hyperparameters (documented magic numbers)
AUGMENTATION_CONFIG = {
    'elastic': {'alpha': 120, 'sigma': 6.0},
    'he_stain': {
        'intensity_scale': (0.8, 1.2),
        'intensity_shift': (-0.1, 0.1),
    },
    'color_jitter': {
        'brightness': 0.2,
        'contrast': 0.2,
        'saturation': 0.2,
        'hue': 0.05,
    },
    'gaussian_noise': {'std_range': (0.01, 0.05)},
}

def get_transforms(size: int = 1024, config: dict | None = None):
    """
    Get training augmentations with tunable hyperparameters.
    
    Args:
        size: Image size
        config: Augmentation config dict. Defaults to AUGMENTATION_CONFIG
    
    Notes:
        Reinhard and Z-score normalization happen in GlomeruliDataset.__getitem__,
        not here. Train transforms run on RGB uint8 images before Z-score; validation
        uses an explicit no-op transform for symmetry.
    """
    if config is None:
        config = AUGMENTATION_CONFIG
    
    train_augment = A.Compose([
        # Geometric augmentations — applied to both image AND mask
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.75),
        A.Transpose(p=0.5),
        A.Rotate(limit=15, p=0.3),
        A.ShiftScaleRotate(scale_limit=0.15, rotate_limit=15, shift_limit=0.1, p=0.5),
        
        # Elastic deformations — simulate tissue preparation artifacts
        A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
        
        # H&E stain augmentation — simulates staining variability between biopsies
        # ImageOnlyTransform: applied to image only, mask is unchanged
        A.HEStain(
            method='random_preset',
            intensity_scale_range=(0.8, 1.2),
            intensity_shift_range=(-0.1, 0.1),
            augment_background=False,
            p=0.4,
        ),

        # Color augmentations — simulate stain variation between slides
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.3),
        
        # Noise — simulate scanner artifacts
        A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
        
        # Regularization via occlusion (use fill=128 to avoid NaN in BatchNorm)
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(32, 64),
            hole_width_range=(32, 64),
            fill=128,
            p=0.2,
        ),
    ], additional_targets={'mask': 'mask'})
    
    val_transform = A.Compose([])
    
    return train_augment, val_transform

# ===== CENTRALIZED POST-PROCESSING FUNCTION =====
# This function is used by threshold tuning, crop extraction, and reconstruction viz
# to ensure CONSISTENCY across all stages

def postprocess_prob_to_instances(
    prob_map,           # H×W float32 probability map (0-1)
    threshold=0.4,
    min_area_px=1500,
    min_distance=15,    # distance between watershed peaks
):
    """
    Consistent post-processing pipeline for all glomerulus detection tasks.
    
    Args:
        prob_map: H×W float32 array in [0, 1]
        threshold: decision threshold
        min_area_px: minimum instance area in pixels
        min_distance: minimum distance between watershed peaks
    
    Returns:
        list of skimage regionprops objects (instances)
    """
    # Step 1: Binary mask from probability
    binary_mask = (prob_map > threshold).astype(np.uint8)
    
    # Step 2: Morphological cleaning (close → open)
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel_close)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel_open)
    
    # Step 3: Watershed-based instance separation
    if binary_mask.sum() < min_area_px:
        return []
    
    distance = distance_transform_edt(binary_mask)
    coords = _plm(distance, min_distance=min_distance, labels=binary_mask)
    
    if len(coords) == 0:
        # Single large region
        coords = np.array([[binary_mask.shape[0]//2, binary_mask.shape[1]//2]])
    
    marker_mask = np.zeros_like(binary_mask, dtype=bool)
    marker_mask[tuple(coords.T)] = True
    markers, _ = scipy_label(marker_mask)
    
    labels_map = _watershed(-distance, markers, mask=binary_mask)
    props = regionprops(labels_map)
    
    # Step 4: Filter by minimum area
    return [p for p in props if p.area >= min_area_px]


In [ ]:
def perturb_mask(binary_mask: np.ndarray) -> np.ndarray:
    """Perturba la máscara binaria para simular error de U-Net en entrenamiento."""
    ops = random.choices(['dilate', 'erode', 'close', 'open', 'shift', 'none'], k=1)[0]
    k = random.choice([3, 5])
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    if ops == 'dilate':
        return cv2.dilate(binary_mask.astype(np.uint8), kernel).astype(bool)
    elif ops == 'erode':
        return cv2.erode(binary_mask.astype(np.uint8), kernel).astype(bool)
    elif ops == 'close':
        return cv2.morphologyEx(binary_mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel).astype(bool)
    elif ops == 'open':
        return cv2.morphologyEx(binary_mask.astype(np.uint8), cv2.MORPH_OPEN, kernel).astype(bool)
    elif ops == 'shift':
        dx, dy = random.randint(-5, 5), random.randint(-5, 5)
        M = np.float32([[1, 0, dx], [0, 1, dy]])
        shifted = cv2.warpAffine(binary_mask.astype(np.uint8), M, binary_mask.shape[::-1])
        return shifted.astype(bool)
    return binary_mask

In [ ]:
class GlomeruliClassificationDataset(Dataset):
    """
    Dataset para clasificación de glomerulos.
    
    Carga tiles crudos desde Entradas/, extrae crops on-the-fly per instancia,
    aplica augmentations, y devuelve [4, 224, 224] tensors (RGB 3 canales preprocesado + máscara binaria 1 canal).
    
    Workflow:
    1. Filtra manifest por split (train/val/test)
    2. Para cada instancia:
       a. Carga tile RGB completo y máscara multiclase
       b. Extrae crop vía bbox con margen
       c. Binariza máscara (any(class_mask) > 0)
       d. Perturba máscara aleatoriamente (simula error de U-Net) solo en train
       e. Resize a input_size
       f. Aplica augmentations geométricas (si configuradas)
       g. Preprocesa RGB: Reinhard + Z-score normalization
       h. Concatena RGB [3, H, W] + máscara [1, H, W] → [4, H, W]
    3. Retorna (x, y) donde x es tensor 4-channel, y es class_id (0-3)
    
    Incluye método get_sampling_weights() para balancear clases en DataLoader.
    """

    def __init__(
        self,
        manifest: pd.DataFrame,
        split: str,
        input_size: int = 224,
        margin_ratio: float = 0.25,
        reinhard_norm=None,
        channel_means: list = None,
        channel_stds: list = None,
        transforms=None,
        mask_perturb_prob: float = 0.5,
        background_value: int = 255,
    ):
        """
        Args:
            manifest: pd.DataFrame con columnas: tile_path, mask_path, bbox_r1/c1/r2/c2, class_id, split
            split: 'train', 'val', o 'test'
            input_size: tamaño del crop resizeado (224)
            margin_ratio: margen adicional alrededor del bbox (fracción del bbox size)
            reinhard_norm: ReinhardNormalize instance o None
            channel_means: [R, G, B] means para Z-score
            channel_stds: [R, G, B] stds para Z-score
            transforms: albumentations Compose con additional_targets={'mask': 'mask'} o None
            mask_perturb_prob: probabilidad de perturbar máscara en split=train (default 0.5)
            background_value: valor para enmascarar fondo (default 255 = blanco)
        """
        self.df = manifest[manifest['split'] == split].reset_index(drop=True)
        self.split = split
        self.input_size = input_size
        self.margin_ratio = margin_ratio
        self.reinhard_norm = reinhard_norm
        self.channel_means = channel_means
        self.channel_stds = channel_stds
        self.transforms = transforms
        self.mask_perturb_prob = mask_perturb_prob
        self.background_value = background_value

    def __len__(self) -> int:
        """Retorna cantidad de instancias en el split."""
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Carga y preprocesa una instancia de glomerulo.
        
        Returns:
            (x, y) donde:
                x: torch.Tensor de shape [4, 224, 224] (float32)
                   - canales 0-2: RGB preprocesado (Reinhard + Z-score)
                   - canal 3: máscara binaria 0-1
                y: torch.Tensor scalar (int64) con class_id ∈ [0, 3]
        """
        row = self.df.iloc[idx]
        
        # ========== 1. Cargar tile RGB completo y máscara multiclase ==========
        img = load_rgb_image(row['tile_path'])           # [H, W, 3] uint8 RGB
        mask_gray = cv2.imread(str(row['mask_path']), cv2.IMREAD_GRAYSCALE)  # [H, W] uint8
        if mask_gray is None:
            raise RuntimeError(f"Failed to load mask: {row['mask_path']}")
        
        # ========== 2. Extraer crop vía bbox con margen ==========
        r1, c1, r2, c2 = int(row['bbox_r1']), int(row['bbox_c1']), int(row['bbox_r2']), int(row['bbox_c2'])
        height = r2 - r1
        width = c2 - c1
        margin = int(max(height, width) * self.margin_ratio)
        
        # Clip al rango de imagen
        r1m = max(0, r1 - margin)
        r2m = min(img.shape[0], r2 + margin)
        c1m = max(0, c1 - margin)
        c2m = min(img.shape[1], c2 + margin)
        
        img_crop = img[r1m:r2m, c1m:c2m].copy()           # [H', W', 3] uint8
        mask_crop_gray = mask_gray[r1m:r2m, c1m:c2m].copy()  # [H', W'] uint8
        
        # ========== 3. Binarizar máscara (any non-zero class) ==========
        mask_crop = (mask_crop_gray > 0).astype(np.uint8)  # [H', W'] uint8 0-1
        
        # ========== 4. Perturbar máscara (solo en split='train') ==========
        if self.split == 'train' and random.random() < self.mask_perturb_prob:
            mask_crop = perturb_mask(mask_crop.astype(bool)).astype(np.uint8)
        
        # ========== 5. Resize a input_size ==========
        img_crop = cv2.resize(img_crop, (self.input_size, self.input_size), interpolation=cv2.INTER_LINEAR)
        mask_crop = cv2.resize(mask_crop, (self.input_size, self.input_size), interpolation=cv2.INTER_NEAREST)
        
        # ========== 6. Enmascarar fondo (poner background_value) ==========
        bg_mask = mask_crop == 0
        img_crop[bg_mask] = self.background_value
        
        # ========== 7. Augmentations geométricas (imagen + máscara) ==========
        if self.transforms is not None:
            augmented = self.transforms(image=img_crop.astype(np.uint8), mask=mask_crop)
            img_crop = augmented['image']
            mask_crop = augmented['mask']
        
        # ========== 8. Preprocesamiento: Reinhard + Z-score (solo RGB, no máscara) ==========
        img_preprocessed = preprocess_rgb_image(
            img_crop,
            self.reinhard_norm,
            self.channel_means,
            self.channel_stds,
        )  # [H, W, 3] float32 [-1, 1] approx
        
        # ========== 9. Construir tensor [4, H, W]: RGB preprocesado + máscara ==========
        img_tensor = torch.from_numpy(img_preprocessed.transpose(2, 0, 1)).float()  # [3, H, W]
        mask_tensor = torch.from_numpy(mask_crop[None].astype(np.float32))  # [1, H, W]
        x = torch.cat([img_tensor, mask_tensor], dim=0)  # [4, H, W]
        
        # ========== 10. Label ==========
        y = torch.tensor(int(row['class_id']), dtype=torch.long)
        
        return x, y

    def get_sampling_weights(self) -> torch.Tensor:
        """
        Retorna pesos para WeightedRandomSampler.
        
        Usa inverse square root de class frequency para balancear clases desbalanceadas.
        Formula: weight_c = 1 / sqrt(count_c)
        
        Returns:
            torch.Tensor de shape [len(self)] con un peso por muestra
        """
        labels = self.df['class_id'].values
        counts = np.bincount(labels, minlength=4)
        class_weights = 1.0 / np.sqrt(counts + 1)  # +1 para evitar div by 0
        sample_weights = torch.tensor(class_weights[labels], dtype=torch.float)
        return sample_weights


In [ ]:
def get_classification_transforms(size: int = 224, config: dict = None):
    """
    Retorna (train_augment, val_transform) para clasificación.
    
    Train: augmentaciones geométricas + color + stain (con p reducida para distorsiones)
    Val: sin augmentaciones
    
    Args:
        size: Tamaño de la imagen (default 224)
        config: Dict de configuración (no usado por ahora, para futuras extensiones)
    
    Returns:
        (train_aug, val_aug): Tupla de albumentations Compose pipelines
    """
    
    train_aug = A.Compose([
        # Transformaciones geométricas — aplican a AMBOS 'image' y 'mask'
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Transpose(p=0.5),
        A.Rotate(limit=15, p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.3),
        
        # Distorsiones elásticas — REDUCIDO: p=0.10 vs 0.25 en U-Net
        # En clasificación queremos evitar distorsiones extremas que pierdan rasgos morfológicos
        A.ElasticTransform(alpha=60, sigma=3.0, p=0.10),
        A.GridDistortion(num_steps=5, distort_limit=0.15, p=0.10),
        
        # Transformaciones de color/tinción — aplican SOLO a 'image', no a 'mask'
        # HEStain simula variación de tinción H&E entre biopsias
        A.HEStain(intensity_scale=(0.8, 1.2), intensity_shift=(-0.1, 0.1), p=0.5),
        
        # Augmentaciones de color
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.4),
        
        # Ruido gaussiano — simula artefactos de scanner
        A.GaussNoise(std_range=(0.01, 0.05), p=0.3),
        
        # Oclusión aleatoria (CoarseDropout)
        A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),
    ], additional_targets={'mask': 'mask'})  # 'mask' pasa por transforms geométricas
    
    val_aug = A.Compose([])  # No-op para validación
    
    return train_aug, val_aug


# ============================================================================
# CELL 8: build_classifier()
# ============================================================================

def build_classifier(
    backbone: str = 'efficientnet_b0',
    num_classes: int = 4,
    in_chans: int = 4,
    pretrained: bool = True,
) -> nn.Module:
    '''
    Crea un modelo de clasificación usando timm.
    EfficientNet-B0 con 4 canales de entrada (RGB + máscara).
    
    Args:
        backbone: nombre del modelo timm (default 'efficientnet_b0')
        num_classes: cantidad de clases (default 4)
        in_chans: canales de entrada (default 4 = RGB 3 + máscara 1)
        pretrained: usar pesos preentrenados en ImageNet (default True)
    
    Returns:
        nn.Module: modelo de clasificación
    
    Nota: timm.create_model() con in_chans=4 modifica automáticamente
    la primera capa convolucional para aceptar 4 canales en lugar de 3.
    No necesita código adicional para adaptar la primera capa.
    '''
    model = timm.create_model(
        backbone,
        pretrained=pretrained,
        num_classes=num_classes,
        in_chans=in_chans,  # timm adapta conv1 automáticamente para 4 canales
    )
    return model

In [ ]:
# ============================================================================
# CELL 10: build_criterion()
# ============================================================================

def build_criterion(manifest: pd.DataFrame, device) -> nn.Module:
    '''
    Crea CrossEntropyLoss con pesos de clase para balanceo.
    
    Los pesos se calculan como: weight_c = 1 / sqrt(count_c)
    Esta fórmula suaviza el balanceo para evitar sobrepeso en clases muy pequeñas.
    
    Args:
        manifest: pd.DataFrame con columna 'class_id' y 'split'
        device: torch.device (cuda o cpu)
    
    Returns:
        nn.Module: CrossEntropyLoss con pesos de clase
    
    Nota: Los pesos se calculan una sola vez en training (no se adaptan por epoch).
    Si Esclerosado/Excluido siguen con recall bajo después de entrenamiento,
    considerar FocalLoss en futuro.
    '''
    
    # Contar instancias por clase en train split
    train_manifest = manifest[manifest['split'] == 'train']
    counts = train_manifest['class_id'].value_counts().sort_index().values
    
    # Pesos: 1 / sqrt(count) para suavizar
    class_weights = 1.0 / np.sqrt(counts + 1)
    
    # Normalizar
    class_weights = class_weights / class_weights.sum() * len(counts)
    
    # CrossEntropyLoss con pesos
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(class_weights, dtype=torch.float).to(device),
        reduction='mean',
    )
    
    return criterion

In [ ]:
# ============================================================================
# CELL 9: create_classification_dataloaders()
# ============================================================================

def create_classification_dataloaders(
    manifest: pd.DataFrame,
    config: dict,
    reinhard_norm=None,
    channel_means: list = None,
    channel_stds: list = None,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    '''
    Crea DataLoaders para train, val, test.
    Train usa WeightedRandomSampler para balanceo de clases.
    
    Args:
        manifest: pd.DataFrame con columnas: tile_path, mask_path, bbox_*, class_id, split
        config: dict con keys: 'input_size', 'margin_ratio', 'mask_perturb_prob', 'batch_size', 'num_workers'
        reinhard_norm: ReinhardNormalize instance o None
        channel_means: [R, G, B] means para Z-score normalization
        channel_stds: [R, G, B] stds para Z-score normalization
    
    Returns:
        (train_loader, val_loader, test_loader): Tupla de DataLoaders
    '''
    
    # Transformations
    train_aug, val_aug = get_classification_transforms(config['input_size'])
    
    # Datasets
    train_ds = GlomeruliClassificationDataset(
        manifest,
        split='train',
        input_size=config['input_size'],
        margin_ratio=config['margin_ratio'],
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
        transforms=train_aug,
        mask_perturb_prob=config['mask_perturb_prob'],
    )
    
    val_ds = GlomeruliClassificationDataset(
        manifest,
        split='val',
        input_size=config['input_size'],
        margin_ratio=config['margin_ratio'],
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
        transforms=val_aug,
        mask_perturb_prob=0.0,  # sin perturbación en validation
    )
    
    test_ds = GlomeruliClassificationDataset(
        manifest,
        split='test',
        input_size=config['input_size'],
        margin_ratio=config['margin_ratio'],
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
        transforms=val_aug,
        mask_perturb_prob=0.0,  # sin perturbación en test
    )
    
    # Sampler para train (balancea clases)
    weights = train_ds.get_sampling_weights()
    sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)
    
    # DataLoaders
    num_workers = min(config.get('num_workers', 4), os.cpu_count() or 1)
    
    train_loader = DataLoader(
        train_ds,
        batch_size=config['batch_size'],
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=True,
    )
    
    val_loader = DataLoader(
        val_ds,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    
    test_loader = DataLoader(
        test_ds,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    
    return train_loader, val_loader, test_loader

In [ ]:
def train_epoch_clf(
    model,
    dataloader,
    criterion,
    optimizer,
    device,
    scaler=None,
    epoch=0,
    log_interval=20,
) -> float:
    '''Entrena 1 epoch. Retorna loss promedio.'''
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    pbar = tqdm(dataloader, desc=f'Train Epoch {epoch}', leave=False)
    
    for batch_idx, (x, y) in enumerate(pbar):
        x, y = x.to(device), y.to(device)
        
        # Forward (con AMP si scaler está disponible)
        if scaler is not None:
            with autocast():
                logits = model(x)
                loss = criterion(logits, y)
        else:
            logits = model(x)
            loss = criterion(logits, y)
        
        # Backward
        optimizer.zero_grad()
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        if batch_idx % log_interval == 0:
            pbar.set_postfix({'loss': loss.item():.4f})
    
    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    return avg_loss


def eval_epoch_clf(
    model,
    dataloader,
    criterion,
    device,
    split_name='val',
) -> Tuple[float, dict]:
    '''Evalúa 1 epoch. Retorna (loss_promedio, metrics_dict).'''
    model.eval()
    total_loss = 0.0
    num_batches = 0
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc=f'Eval {split_name}', leave=False)
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            
            logits = model(x)
            loss = criterion(logits, y)
            
            total_loss += loss.item()
            num_batches += 1
            
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
    
    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    
    # Concatenar resultados
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    
    # Métricas
    from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix, precision_recall_fscore_support
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    
    # Per-class recall y precision
    precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
        all_labels, all_preds, average=None, zero_division=0
    )
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
    
    metrics_dict = {
        'loss': avg_loss,
        'balanced_accuracy': balanced_acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'f1_per_class': f1_per_class,
        'confusion_matrix': cm,
        'all_preds': all_preds,
        'all_labels': all_labels,
        'all_probs': all_probs,
    }
    
    return avg_loss, metrics_dict

In [ ]:
def export_multiclass_mask(
    tile_rgb_path: str,
    classifier_model,
    unet_prob_map: np.ndarray,
    unet_threshold: float,
    device,
    config: dict,
    reinhard_norm,
    channel_means: list,
    channel_stds: list,
    output_dir: str = None,
) -> Tuple[np.ndarray, list]:
    '''
    Clasifica instancias en tile y exporta PNG multiclass.
    
    Args:
        tile_rgb_path: ruta al tile RGB original (1024×1024)
        classifier_model: modelo de clasificación entrenado
        unet_prob_map: [1024, 1024] float32, salida de U-Net (0-1)
        unet_threshold: threshold para binarizar la salida de U-Net
        device: torch device
        config: CONFIG dict
        reinhard_norm, channel_means, channel_stds: preprocessing params
        output_dir: directorio para guardar PNG y CSV
    
    Returns:
        pred_mask_gray: [1024, 1024] uint8 con valores {0, 64, 128, 192, 255}
        instances_report: list de dicts con info de cada instancia
    '''
    
    tile_rgb = load_rgb_image(tile_rgb_path)  # [1024, 1024, 3] uint8
    
    # Detección de instancias a partir de mapa de U-Net
    instances = postprocess_prob_to_instances(
        unet_prob_map,
        threshold=unet_threshold,
        min_area_px=config['min_area_px'],
        min_distance=config['min_distance'],
    )
    
    # Inicializar máscara de salida
    pred_mask_gray = np.zeros((1024, 1024), dtype=np.uint8)
    
    instances_report = []
    classifier_model.eval()
    
    with torch.no_grad():
        for inst_id, instance in enumerate(instances):
            r1, c1, r2, c2 = instance.bbox
            
            # Margin
            height = r2 - r1
            width = c2 - c1
            margin = int(max(height, width) * config['margin_ratio'])
            
            r1m = max(0, r1 - margin)
            r2m = min(tile_rgb.shape[0], r2 + margin)
            c1m = max(0, c1 - margin)
            c2m = min(tile_rgb.shape[1], c2 + margin)
            
            # Extraer crop
            img_crop = tile_rgb[r1m:r2m, c1m:c2m].copy()
            instance_mask = instance.image.astype(np.uint8)  # [H', W'] binary
            
            # Resize
            img_crop = cv2.resize(img_crop, (config['input_size'], config['input_size']), interpolation=cv2.INTER_LINEAR)
            mask_crop = cv2.resize(instance_mask, (config['input_size'], config['input_size']), interpolation=cv2.INTER_NEAREST)
            
            # Enmascarar fondo
            bg_mask = mask_crop == 0
            img_crop[bg_mask] = 255
            
            # Preprocesar
            img_preprocessed = preprocess_rgb_image(
                img_crop,
                reinhard_norm,
                channel_means,
                channel_stds,
            )
            
            # Tensor [4, 224, 224]
            img_tensor = torch.from_numpy(img_preprocessed.transpose(2, 0, 1)).float()
            mask_tensor = torch.from_numpy(mask_crop[None].astype(np.float32))
            x = torch.cat([img_tensor, mask_tensor], dim=0).to(device)[None]  # [1, 4, 224, 224]
            
            # Clasificar
            logits = classifier_model(x)
            probs = torch.softmax(logits, dim=1)
            pred_class_id = torch.argmax(logits, dim=1).item()
            pred_prob = probs[0, pred_class_id].item()
            
            # Mapear a valor gris
            gray_value = CLASS_TO_GRAY[pred_class_id]
            
            # Rellenar máscara de salida
            pred_mask_gray[r1:r2, c1:c2][instance_mask > 0] = gray_value
            
            # Reportar
            instances_report.append({
                'instance_id': inst_id,
                'class_id': pred_class_id,
                'class_name': CLASS_NAMES[pred_class_id],
                'gray_value': gray_value,
                'probability': pred_prob,
                'bbox': (r1, c1, r2, c2),
                'area_px': np.sum(instance_mask > 0),
            })
    
    # Guardar PNG
    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        png_path = output_dir / f'{Path(tile_rgb_path).stem}_classified.png'
        cv2.imwrite(str(png_path), pred_mask_gray)
        
        # Guardar CSV
        csv_path = output_dir / f'{Path(tile_rgb_path).stem}_instances.csv'
        df_instances = pd.DataFrame(instances_report)
        df_instances.to_csv(csv_path, index=False)
    
    return pred_mask_gray, instances_report

In [ ]:
def evaluate_pipeline(
    test_manifest: pd.DataFrame,
    classifier_model,
    device,
    config: dict,
    reinhard_norm,
    channel_means: list,
    channel_stds: list,
) -> dict:
    '''
    Evalúa el pipeline completo comparando:
    - Clasificación con máscaras manuales (Ground Truth)
    vs.
    - Clasificación con máscaras de U-Net (predicción)
    
    Reporta mIoU y Dice per-class entre PNG predicho y PNG original.
    '''
    
    classifier_model.eval()
    
    # Funciones de utilidad para mIoU y Dice
    def compute_iou_per_class(pred, gt, num_classes):
        """Computa IoU por clase: I(pred==c, gt==c) / U(pred==c | gt==c)"""
        iou_per_class = []
        for c in range(num_classes):
            pred_c = pred == c
            gt_c = gt == c
            intersection = np.sum(pred_c & gt_c)
            union = np.sum(pred_c | gt_c)
            iou = intersection / union if union > 0 else 0.0
            iou_per_class.append(iou)
        return iou_per_class
    
    def compute_dice_per_class(pred, gt, num_classes):
        """Computa Dice (F1) por clase: 2*I / (|pred| + |gt|)"""
        dice_per_class = []
        for c in range(num_classes):
            pred_c = pred == c
            gt_c = gt == c
            intersection = np.sum(pred_c & gt_c)
            dice = 2 * intersection / (np.sum(pred_c) + np.sum(gt_c)) if (np.sum(pred_c) + np.sum(gt_c)) > 0 else 0.0
            dice_per_class.append(dice)
        return dice_per_class
    
    # Evaluar en test set
    test_df = test_manifest[test_manifest['split'] == 'test'].reset_index(drop=True)
    
    all_ious = {c: [] for c in range(NUM_CLASSES)}
    all_dices = {c: [] for c in range(NUM_CLASSES)}
    
    with torch.no_grad():
        for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Evaluating pipeline'):
            # Cargar tile y mascaras
            tile_rgb = load_rgb_image(row['tile_path'])
            mask_gt = load_binary_mask(row['mask_path'], binary=False)  # multiclass
            
            # Extraer instancia via bbox
            r1, c1, r2, c2 = row['bbox_r1'], row['bbox_c1'], row['bbox_r2'], row['bbox_c2']
            height = r2 - r1
            width = c2 - c1
            margin = int(max(height, width) * config['margin_ratio'])
            
            r1m = max(0, r1 - margin)
            r2m = min(tile_rgb.shape[0], r2 + margin)
            c1m = max(0, c1 - margin)
            c2m = min(tile_rgb.shape[1], c2 + margin)
            
            img_crop = tile_rgb[r1m:r2m, c1m:c2m]
            mask_gt_crop = mask_gt[r1m:r2m, c1m:c2m]  # [H', W'] valores 0/64/128/192/255
            
            # Binarizar GT para extraer instancia
            mask_binary = (mask_gt_crop > 0).astype(np.uint8)
            
            # Resize
            img_crop = cv2.resize(img_crop, (config['input_size'], config['input_size']), interpolation=cv2.INTER_LINEAR)
            mask_binary = cv2.resize(mask_binary, (config['input_size'], config['input_size']), interpolation=cv2.INTER_NEAREST)
            
            # Enmascarar fondo
            bg_mask = mask_binary == 0
            img_crop[bg_mask] = 255
            
            # Preprocesar
            img_preprocessed = preprocess_rgb_image(img_crop, reinhard_norm, channel_means, channel_stds)
            
            # Tensor
            img_tensor = torch.from_numpy(img_preprocessed.transpose(2, 0, 1)).float()
            mask_tensor = torch.from_numpy(mask_binary[None].astype(np.float32))
            x = torch.cat([img_tensor, mask_tensor], dim=0).to(device)[None]  # [1, 4, 224, 224]
            
            # Clasificar
            logits = classifier_model(x)
            pred_class_id = torch.argmax(logits, dim=1).item()
            pred_gray = CLASS_TO_GRAY[pred_class_id]
            
            # Crear mapas de comparación (ambos en escala de clases 0-3)
            pred_mask_class = np.ones_like(mask_binary) * pred_class_id  # background = pred_class_id (wrong, but for Dice)
            pred_mask_class[mask_binary == 0] = 0  # background = 0 en pred
            
            gt_mask_class = np.zeros_like(mask_gt_crop)  # inicializar en background (0)
            gt_instance = mask_gt_crop > 0
            gt_class_id = GRAY_TO_CLASS[np.bincount(mask_gt_crop.flatten()).argmax()]
            gt_mask_class[gt_instance] = max(0, gt_class_id)  # evitar -1 (background)
            
            # Computar IoU y Dice solo en región de instancia
            iou_per_class = compute_iou_per_class(pred_mask_class, gt_mask_class, NUM_CLASSES)
            dice_per_class = compute_dice_per_class(pred_mask_class, gt_mask_class, NUM_CLASSES)
            
            for c in range(NUM_CLASSES):
                all_ious[c].append(iou_per_class[c])
                all_dices[c].append(dice_per_class[c])
    
    # Promediar
    miou_per_class = {c: np.mean(all_ious[c]) for c in range(NUM_CLASSES)}
    dice_per_class = {c: np.mean(all_dices[c]) for c in range(NUM_CLASSES)}
    miou_overall = np.mean([miou_per_class[c] for c in range(NUM_CLASSES)])
    dice_overall = np.mean([dice_per_class[c] for c in range(NUM_CLASSES)])
    
    # Reporte
    print("\\n" + "="*60)
    print("PIPELINE EVALUATION (Predictions vs Ground Truth)")
    print("="*60)
    print(f"\\nOverall mIoU: {miou_overall:.4f}")
    print(f"Overall Dice: {dice_overall:.4f}")
    print(f"\\nPer-Class Metrics:")
    for c in range(NUM_CLASSES):
        print(f"  {CLASS_NAMES[c]:20s}: mIoU={miou_per_class[c]:.4f} Dice={dice_per_class[c]:.4f}")
    print("="*60)
    
    return {
        'miou_overall': miou_overall,
        'dice_overall': dice_overall,
        'miou_per_class': miou_per_class,
        'dice_per_class': dice_per_class,
    }